%md
# Demo Setup

First read the prerequisites and steps in **`README.md`**.

Simply fill out the widget values above and then run this notebook from top to bottom ("Run all") to set up the demo. If you encounter permission issues, ensure you have access to grant permissions on the Catalog/Schema.

In [0]:
dbutils.widgets.text("catalog", "main", "The catalog to use")
dbutils.widgets.text("schema", "social_listening", "The schema to use")
dbutils.widgets.text("warehouse_id", "", "SQL Warehouse for reporting")

0. Install and init latest Databricks CLI

In [0]:
%sh
# Install latest version from official source
curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

In [0]:
import os
from dbruntime import databricks_repl_context
os.environ['ENABLE_DATABRICKS_CLI']="true"
os.environ['DATABRICKS_TOKEN']=databricks_repl_context.get_context().apiToken
os.environ['DATABRICKS_HOST']=f"https://{databricks_repl_context.get_context().browserHostName}"

In [0]:
%sh
databricks -v

In [0]:
import os, re, requests, subprocess, json

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
warehouse_id = dbutils.widgets.get("warehouse_id")
prefix = re.sub(r'[^a-zA-Z0-9_]', '', spark.sql("SELECT current_user()").collect()[0][0].split("@")[0])[:15]

# Optional: use a prefix besides your username: (lowercase letters, numbers, dashes (-) only)
# prefix = "my-prefix"

import os, requests, subprocess, json

0.5. Populate placeholder secret

Run the Secrets_Helper notebook or use the in-app settings page(s) to populate placeholder secrets for Reddit and Steam API keys.

In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

secret_scope = "social_listening_app"
workspace_client = WorkspaceClient()

# Create the secret scope
try:
    workspace_client.secrets.create_scope(scope=secret_scope)
    print(f"Created secret scope: {secret_scope}")
    
    # Steam
    steam_api_key_value = ""
    # Reddit
    reddit_client_id_value = ""
    reddit_client_secret_value = ""
    reddit_user_agent_value = ""
    
    # Add place holder secrets to the scope
    workspace_client.secrets.put_secret(scope=secret_scope, key="steam_api_key", string_value=steam_api_key_value)
    print("Added steam_api_key")
    workspace_client.secrets.put_secret(scope=secret_scope, key="reddit_client_id", string_value=reddit_client_id_value)
    print("Added reddit_client_id")
    workspace_client.secrets.put_secret(scope=secret_scope, key="reddit_client_secret", string_value=reddit_client_secret_value)
    print("Added reddit_client_secret")
    workspace_client.secrets.put_secret(scope=secret_scope, key="reddit_user_agent", string_value=reddit_user_agent_value)
    print("Added reddit_user_agent")

except ResourceAlreadyExists:
    print(f"Secret scope '{secret_scope}' already exists, continuing...")

1. Deploy Bundle

In [0]:
cmd = (
    "cd bundle && databricks bundle validate "
    + f'--var="catalog={dbutils.widgets.get("catalog")}" '
    + f'--var="schema={dbutils.widgets.get("schema")}" '
    + f'--var="warehouse_id={dbutils.widgets.get("warehouse_id")}" '
    + f'--var="prefix={prefix}"'
)
subprocess.run(["bash", "-lc", cmd], check=True)

In [0]:
cmd = (
    "cd bundle && databricks bundle deploy --auto-approve --output text "
    + f'--var="catalog={catalog}" '
    + f'--var="schema={schema}" '
    + f'--var="warehouse_id={warehouse_id}" '
    + f'--var="prefix={prefix}"'
)
subprocess.run(["bash", "-lc", cmd], check=True)

2. Run Initial Job (8-12mins)

In [0]:
# Only run initial job if no data exists
table_name = f"{catalog}.{schema}.feedback_content_gold"
try:
    row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]
except Exception as ex:
    if hasattr(ex, "getErrorClass") and ex.getErrorClass() == "TABLE_OR_VIEW_NOT_FOUND":
        row_count = 0
    else:
        raise

if row_count == 0:
    cmd = (
        "cd bundle && databricks bundle run games_social_listening_job "
        + f'--var="catalog={catalog}" '
        + f'--var="schema={schema}" '
        + f'--var="warehouse_id={dbutils.widgets.get("warehouse_id")}" '
        + f'--var="prefix={prefix}"'
    )
    subprocess.run(["bash", "-lc", cmd], check=True)
else:
    print(f"WARNING: Skipping initial job: {table_name} already contains data.")

3. Grab Required Variables from DAB via CLI

In [0]:
cmd = (
    "cd bundle && databricks bundle summary "
    + f'--var="catalog={dbutils.widgets.get("catalog")}" '
    + f'--var="schema={dbutils.widgets.get("schema")}" '
    + f'--var="warehouse_id={dbutils.widgets.get("warehouse_id")}" '
    + f'--var="prefix={prefix}"'
    + " --output json"
)
res = subprocess.run(["bash", "-lc", cmd], check=True, capture_output=True, text=True)
summary = json.loads(res.stdout)

# Extract IDs
dashboard_id = summary["resources"]["dashboards"]["games_social_listening_dashboard"]["id"]
print("Dashboard ID:", dashboard_id)

ingestion_job_id = summary["resources"]["jobs"]["games_social_listening_job"]["id"]
print("Ingestion Job ID:", ingestion_job_id)

app_name = summary["resources"]["apps"]["games_social_listening_app"]["name"]
print("App Name:", app_name)

bundle_resource_path = summary["resources"]["dashboards"]["games_social_listening_dashboard"]["parent_path"]
print("Bundle Resource Path:", bundle_resource_path)

4. Create Genie Space

In [0]:
genie_api_endpoint = f"{os.environ['DATABRICKS_HOST']}/api/2.0/genie/spaces"
token = os.environ['DATABRICKS_TOKEN']

description = "Space to analyze the sentiment of user generated content across video games"
parent_path = bundle_resource_path
serialized_space_file = "bundle/src/genie_space/serialized_space.json"
genie_instructions_file = "bundle/src/genie_space/genie_instructions.txt"
title = f"{prefix} - Games Social Listening"

# Check For Existing Genie Space (by title) - use existing space instead of creating a new one
page_token = "*"
all_spaces = []
while page_token:
    list_resp = requests.get(
        genie_api_endpoint,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        params={"page_token": page_token}
    )
    list_resp.raise_for_status()
    spaces = list_resp.json().get("spaces", [])
    all_spaces += spaces
    page_token = list_resp.json().get("next_page_token")

existing_space = next((s for s in all_spaces if s.get("title") == title), None)

if existing_space:
    space_id = existing_space["space_id"]
    print(f"WARNING: skipping Genie space creation, using existing space: {space_id}")
else:
    # Read Genie instructions from text file
    genie_instructions = ""
    with open(genie_instructions_file, "r") as instructions_file:
        genie_instructions_list = instructions_file.read().splitlines(keepends=True)
        genie_instructions = json.dumps(genie_instructions_list)

    with open(serialized_space_file, "r") as f:
        serialized_space = f.read().replace("{UC_SCHEMA}", f"{catalog}.{schema}")
        serialized_space = serialized_space.replace("{GENIE_INSTRUCTIONS}", genie_instructions)

        body = {
            "title": title,
            "description": description,
            "serialized_space": serialized_space,
            "warehouse_id": dbutils.widgets.get("warehouse_id"),
            "parent_path": parent_path
        }
        #dbutils.fs.mkdirs(parent_path)
        resp = requests.post(
            genie_api_endpoint,
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json=body
        )
        if resp.status_code != 200:
            raise Exception(f"Error creating Genie space: {resp.text}")
        else:
            resp_body = resp.json()
            space_id = resp_body['space_id']
            print(f"Genie space created, id: {space_id} title: {resp_body['title']}")

5. Update App Config

In [0]:
# Get values from widgets populated in Job
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema") 
dashboard_url_game_filter_suffix = "f_c2d7b8e7%7E44a4e39e="
dashboard_url_category_page_game_filter_suffix = "&f_fb65c9fa%7E19bdd000="
genie_space_id = space_id

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Ingestion Job ID: {ingestion_job_id}")
print(f"Dashboard ID: {dashboard_id}")
print(f"Genie Space ID: {genie_space_id}")

# Get the host from the workspace context
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
host = w.config.host

# Construct the dashboard URL from the dashboard ID
dashboard_url_base = f"{host}/embed/dashboardsv3/{dashboard_id}?"

print(f"Host: {host}")
print(f"Dashboard URL Base: {dashboard_url_base}")

# Read the app config file
import yaml
import re

config_path = "bundle/src/app/config.yaml"

# Read the config file
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update values
config["databricks"]["host"] = host.replace("https://", "")
config["databricks"]["catalog"] = catalog
config["databricks"]["schema"] = schema
config["databricks"]["ingestion_job_id"] = f"{ingestion_job_id}"
config["databricks"]["dashboard"]["url"]["base"] = f"{dashboard_url_base}"
config["databricks"]["dashboard"]["url"]["game_filter_suffix"] = dashboard_url_game_filter_suffix
config["databricks"]["dashboard"]["url"]["category_page_game_filter_suffix"] = dashboard_url_category_page_game_filter_suffix
config["databricks"]["genie"]["space_id"] = f"{genie_space_id}"
config["databricks"]["secrets"]["scope"] = f"{secret_scope}"
config["ui"]["sidebar"]["link_sections"]["internal"]["links"]["catalog"]["url"] = f"{host}/explore/data/{catalog}/{schema}"
config["ui"]["sidebar"]["link_sections"]["internal"]["links"]["published_dashboard"]["url"] = f"{host}/dashboardsv3/{dashboard_id}/published"
config["ui"]["sidebar"]["link_sections"]["internal"]["links"]["genie_space"]["url"] = f"{host}/genie/rooms/{genie_space_id}"
config["ui"]["sidebar"]["link_sections"]["internal"]["links"]["lakeflow_job"]["url"] = f"{host}/jobs/{ingestion_job_id}"

# Write the updated config back
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("✅ App configuration file updated successfully!")

6. Redeploy the Bundle with the Updates to the app config

In [0]:
cmd = (
    "cd bundle && databricks bundle deploy --auto-approve --output text "
    + f'--var="catalog={dbutils.widgets.get("catalog")}" '
    + f'--var="schema={dbutils.widgets.get("schema")}" '
    + f'--var="warehouse_id={dbutils.widgets.get("warehouse_id")}" '
    + f'--var="prefix={prefix}"'
)
subprocess.run(["bash", "-lc", cmd], check=True)

7. Start app 

In [0]:
cmd = (
    "cd bundle && databricks bundle run  games_social_listening_app "
    + f'--var="catalog={dbutils.widgets.get("catalog")}" '
    + f'--var="schema={dbutils.widgets.get("schema")}" '
    + f'--var="warehouse_id={dbutils.widgets.get("warehouse_id")}" '
    + f'--var="prefix={prefix}"'
)
subprocess.run(["bash", "-lc", cmd], check=True)


8. Provide the app service principal access to the catalog, Genie, and secret scope

In [0]:
from databricks.sdk.service.workspace import AclPermission

resp = requests.get(
    f"{host}/api/2.0/apps/{app_name}",
    headers={"Authorization": f"Bearer {token}"}
)
resp.raise_for_status()
app = resp.json()

app_sp_id = (
    app.get("service_principal_id")
)

# Fetch the service principal name using the ID
sp_resp = requests.get(
    f"{host}/api/2.0/preview/scim/v2/ServicePrincipals/{app_sp_id}",
    headers={"Authorization": f"Bearer {token}"}
)
sp_resp.raise_for_status()
sp_data = sp_resp.json()
app_sp_name = sp_data.get("applicationId")

grant_sql = f"""
GRANT USE CATALOG, USE SCHEMA, SELECT ON CATALOG {catalog} TO `{app_sp_name}`;
"""

spark.sql(grant_sql)

# Grant CAN_RUN on Genie Space using Databricks Workspace Permissions API
genie_permissions_endpoint = f"{host}/api/2.0/permissions/genie/{genie_space_id}"
permissions_body = {
    "access_control_list": [
        {
            "service_principal_name": app_sp_name,
            "permission_level": "CAN_RUN"
        }
    ]
}
permissions_resp = requests.put(
    genie_permissions_endpoint,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json=permissions_body
)
permissions_resp.raise_for_status()

# Grant WRITE permission to the secret scope
workspace_client.secrets.put_acl(
    scope=secret_scope,
    principal=app_sp_name,
    permission=AclPermission.WRITE
)

print(f"✅ Permissions granted to service principal '{app_sp_name}'")
print(f"   - Catalog: {catalog} (USE CATALOG,USE SCHEMA, SELECT)")
print(f"   - Genie Space: {genie_space_id} (CAN_RUN)")
print(f"   - Secret Scope: {secret_scope} (WRITE)")

### DEMO TEAR DOWN
Uncomment out the following cells to tear down the demo.

In [0]:
## Destroy Bundle Resources - rerun CLI install cells if needed
# import subprocess
# cmd = (
#     "cd bundle && databricks bundle destroy --auto-approve "
#     + f'--var="catalog={dbutils.widgets.get("catalog")}" '
#     + f'--var="schema={dbutils.widgets.get("schema")}" '
#     + f'--var="warehouse_id={dbutils.widgets.get("warehouse_id")}" '
#     + f'--var="prefix={prefix}"'
# )
# subprocess.run(["bash", "-lc", cmd], check=True)

In [0]:
## Delete remaining tables
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")

# table_bronze = f"{catalog}.{schema}.feedback_content_bronze"
# table_reports = f"{catalog}.{schema}.feedback_content_reports"

# spark.sql(f"DROP TABLE IF EXISTS {table_bronze}")
# spark.sql(f"DROP TABLE IF EXISTS {table_reports}")

In [0]:
# # Delete secrets and scope
# from databricks.sdk import WorkspaceClient
# from databricks.sdk.errors import ResourceDoesNotExist

# try:
#     workspace_client = WorkspaceClient()
#     workspace_client.secrets.delete_scope(scope=secret_scope)
#     print(f"Deleted secret scope: {secret_scope}")
# except ResourceDoesNotExist:
#     print(f"Secret scope '{secret_scope}' does not exist; nothing to delete.")